<a href="https://colab.research.google.com/github/pgordin/OptDisc2026/blob/main/projekt_algorytmy_wegierskie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Algorytm węgierski - Algorytmy optymalizacji dyskretnej
#Natalia Kostrzewska, nr 263310

#Edukacyjna implementacja algorytmu węgierskiego
#redukcja wierszy, redukcja kolumn, szukanie maksymalnego skojarzenia na zerach,
#pokrywanie zer minimalną liczbą linii oraz korekta macierzy. Wynik jest
#porównywany z biblioteką SciPy oraz z metodą brute force.


import numpy as np
from scipy.optimize import linear_sum_assignment
from itertools import permutations

# WPROWADZANIE MACIERZY
def get_matrix_from_user():
    print("\n=== WPROWADZANIE MACIERZY ===")
    n = int(input("Podaj liczbę wierszy: "))
    m = int(input("Podaj liczbę kolumn: "))

    matrix = []
    for i in range(n):
        row = list(map(int, input(f"Wiersz {i + 1}: ").split()))
        if len(row) != m:
            raise ValueError("Błędna liczba elementów w wierszu!")
        matrix.append(row)

    return np.array(matrix, dtype=float)

# LOSOWA MACIERZ

def generate_random_matrix():
    n = int(input("Podaj liczbę wierszy: "))
    m = int(input("Podaj liczbę kolumn: "))

    matrix = np.random.randint(0, 10, (n, m))
    print("\nWylosowana macierz:")
    print(matrix)

    return matrix.astype(float)


# ALGORYTM WĘGIERSKI (UPROSZCZONY)

class HungarianAlgorithm:

    def __init__(self, matrix):
        self.original_matrix = np.array(matrix, dtype=float)
        self.matrix = np.array(matrix, dtype=float)
        self.rows, self.cols = self.matrix.shape
        self.independent_zeros = []

    def print_matrix(self, title):
        print(f"\n=== {title} ===")
        print(self.matrix)

    #  REDUKCJA WIERSZY
    def reduce_rows(self):
        for i in range(self.rows):
            self.matrix[i] -= np.min(self.matrix[i])
        self.print_matrix("PO REDUKCJI WIERSZY")

    #  REDUKCJA KOLUMN
    def reduce_columns(self):
        for j in range(self.cols):
            self.matrix[:, j] -= np.min(self.matrix[:, j])
        self.print_matrix("PO REDUKCJI KOLUMN")

    #  MAKSYMALNE SKOJARZENIE NA ZERACH (ścieżki powiększające, DFS)
    def _match_zeros(self):
        match_row = [-1] * self.rows
        match_col = [-1] * self.cols

        def augment(i, visited):
            for j in range(self.cols):
                if self.matrix[i][j] == 0 and not visited[j]:
                    visited[j] = True
                    if match_col[j] == -1 or augment(match_col[j], visited):
                        match_row[i] = j
                        match_col[j] = i
                        return True
            return False

        for i in range(self.rows):
            augment(i, [False] * self.cols)

        return match_row, match_col

    #  POKRYWANIE ZER MINIMALNĄ LICZBĄ LINII
    def _cover_zeros(self, match_row, match_col):
        marked_rows = {i for i in range(self.rows) if match_row[i] == -1}
        marked_cols = set()

        changed = True
        while changed:
            changed = False
            for i in list(marked_rows):
                for j in range(self.cols):
                    if self.matrix[i][j] == 0 and j not in marked_cols:
                        marked_cols.add(j)
                        changed = True
            for j in list(marked_cols):
                i = match_col[j]
                if i != -1 and i not in marked_rows:
                    marked_rows.add(i)
                    changed = True

        covered_rows = set(range(self.rows)) - marked_rows
        covered_cols = marked_cols

        print("\n=== POKRYCIE ZER ===")
        print(f"Pokryte wiersze: {sorted(r + 1 for r in covered_rows)}")
        print(f"Pokryte kolumny: {sorted(c + 1 for c in covered_cols)}")
        print(f"Liczba linii: {len(covered_rows) + len(covered_cols)}")

        return covered_rows, covered_cols

    # KOREKTA MACIERZY
    def _correct_matrix(self, covered_rows, covered_cols):
        uncovered = [
            (i, j)
            for i in range(self.rows)
            for j in range(self.cols)
            if i not in covered_rows and j not in covered_cols
        ]
        k = min(self.matrix[i][j] for (i, j) in uncovered)

        print("\n=== KOREKTA MACIERZY ===")
        print("Elementy niepokryte:")
        for (i, j) in uncovered:
            print(f"  ({i + 1}, {j + 1}) = {self.matrix[i][j]}")
        print(f"Minimalny element niepokryty k = {k}")

        for i in range(self.rows):
            for j in range(self.cols):
                if i not in covered_rows and j not in covered_cols:
                    self.matrix[i][j] -= k
                elif i in covered_rows and j in covered_cols:
                    self.matrix[i][j] += k

        self.print_matrix("PO KOREKCIE")

    # WYBÓR NIEZALEŻNYCH ZER (pełna pętla: skojarzenie -> pokrycie -> korekta)
    def find_independent_zeros(self):
        target = min(self.rows, self.cols)

        while True:
            match_row, match_col = self._match_zeros()
            if sum(1 for j in match_row if j != -1) >= target:
                break
            covered_rows, covered_cols = self._cover_zeros(match_row, match_col)
            self._correct_matrix(covered_rows, covered_cols)

        self.independent_zeros = [
            (i, match_row[i]) for i in range(self.rows) if match_row[i] != -1
        ]

        print("\n=== WYBÓR NIEZALEŻNYCH ZER ===")
        for (i, j) in self.independent_zeros:
            print(f"Wybrano: ({i}, {j})")

        cost = sum(self.original_matrix[i][j] for (i, j) in self.independent_zeros)
        print("\nPrzypisanie (algorytm węgierski):")
        for (i, j) in self.independent_zeros:
            print(f"Wiersz {i + 1} -> Kolumna {j + 1} | koszt = {self.original_matrix[i][j]}")
        print(f"\nMinimalny koszt = {cost}")

        return self.independent_zeros

    # BRUTE FORCE (PEŁNE ROZWIĄZANIE)
    def brute_force_solution(self):
        print("\n=== BRUTE FORCE (PEŁNE ROZWIĄZANIE) ===")
        best_cost = float("inf")
        best_perm = None

        for perm in permutations(range(self.cols), self.rows):
            cost = sum(self.original_matrix[i][perm[i]] for i in range(self.rows))
            if cost < best_cost:
                best_cost = cost
                best_perm = perm

        for i in range(self.rows):
            j = best_perm[i]
            print(f"Wiersz {i + 1} -> Kolumna {j + 1} | koszt = {self.original_matrix[i][j]}")
        print(f"\nMinimalny koszt = {best_cost}")

        return best_perm, best_cost

    # SCIPY (PROFESJONALNY ALGORYTM PRZYPISANIA)
    def scipy_solution(self):
        print("\n=== ROZWIĄZANIE SCIPY ===")
        row_ind, col_ind = linear_sum_assignment(self.original_matrix)
        cost = self.original_matrix[row_ind, col_ind].sum()

        for r, c in zip(row_ind, col_ind):
            print(f"Wiersz {r + 1} -> Kolumna {c + 1} | koszt = {self.original_matrix[r][c]}")
        print(f"\nMinimalny koszt (SciPy) = {cost}")

        return cost
# MENU GŁÓWNE

def menu():
    print("\n-------------------------")
    print(" ALGORYTM WĘGIERSKI")
    print("------------------------")
    print("1. Wprowadź macierz ręcznie")
    print("2. Losowa macierz")
    print("3. Wyjście")

    choice = input("Wybierz opcję: ")

    if choice == "1":
        return get_matrix_from_user()
    elif choice == "2":
        return generate_random_matrix()
    else:
        exit()


if __name__ == "__main__":
    matrix = menu()
    solver = HungarianAlgorithm(matrix)

    print("\nMACIERZ WEJŚCIOWA:")
    print(matrix)

    solver.reduce_rows()
    solver.reduce_columns()
    solver.find_independent_zeros()
    solver.brute_force_solution()
    solver.scipy_solution()



-------------------------
 ALGORYTM WĘGIERSKI
------------------------
1. Wprowadź macierz ręcznie
2. Losowa macierz
3. Wyjście
Wybierz opcję: 2
Podaj liczbę wierszy: 3
Podaj liczbę kolumn: 3

Wylosowana macierz:
[[9 3 4]
 [1 1 5]
 [3 0 4]]

MACIERZ WEJŚCIOWA:
[[9. 3. 4.]
 [1. 1. 5.]
 [3. 0. 4.]]

=== PO REDUKCJI WIERSZY ===
[[6. 0. 1.]
 [0. 0. 4.]
 [3. 0. 4.]]

=== PO REDUKCJI KOLUMN ===
[[6. 0. 0.]
 [0. 0. 3.]
 [3. 0. 3.]]

=== WYBÓR NIEZALEŻNYCH ZER ===
Wybrano: (0, 2)
Wybrano: (1, 0)
Wybrano: (2, 1)

Przypisanie (algorytm węgierski):
Wiersz 1 -> Kolumna 3 | koszt = 4.0
Wiersz 2 -> Kolumna 1 | koszt = 1.0
Wiersz 3 -> Kolumna 2 | koszt = 0.0

Minimalny koszt = 5.0

=== BRUTE FORCE (PEŁNE ROZWIĄZANIE) ===
Wiersz 1 -> Kolumna 3 | koszt = 4.0
Wiersz 2 -> Kolumna 1 | koszt = 1.0
Wiersz 3 -> Kolumna 2 | koszt = 0.0

Minimalny koszt = 5.0

=== ROZWIĄZANIE SCIPY ===
Wiersz 1 -> Kolumna 3 | koszt = 4.0
Wiersz 2 -> Kolumna 1 | koszt = 1.0
Wiersz 3 -> Kolumna 2 | koszt = 0.0

Minimalny koszt 